# Playground Series S5E7: Phase 4 Enhanced Integration

**⚠️ OVERFITTING CASE STUDY: What NOT to do in feature engineering**

This notebook implements the Phase 4 Enhanced Integration approach that achieved:
- **CV Score**: 0.978715 (highest ever achieved)
- **PB Score**: 0.919028 (catastrophic failure)
- **CV-PB Gap**: -0.059687 (85x worse than Phase 3)

## ⚠️ Warning: Educational Purpose Only

This implementation serves as a **critical learning example** of how sophisticated feature engineering can lead to severe overfitting. While the CV score is impressive, the Public Board score demonstrates a complete failure to generalize.

## Key Anti-Patterns Demonstrated
1. **Feature Explosion**: 17→41 features (141% increase)
2. **Over-Engineering**: Complex N-gram + Target Encoding combinations
3. **Sophisticated Imputation**: KNN + personality-aware filling
4. **Ignoring Simplicity**: More complex ≠ better performance

## Learning Objectives
- Understand how overfitting manifests in practice
- Recognize warning signs of excessive complexity
- Appreciate the value of simpler approaches
- Learn from failure to improve future implementations

**Author**: Osawa  
**Date**: 2025-07-04  
**Purpose**: Educational case study of severe overfitting

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import VotingClassifier
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import itertools
import warnings
warnings.filterwarnings('ignore')

print("⚠️  WARNING: This is an overfitting demonstration!")
print("📚 Educational purpose: Learn what NOT to do in feature engineering")
print("🎯 Expected result: High CV (0.978715) but low PB (0.919028)")

⚠️  WARNING: This is an overfitting demonstration!
📚 Educational purpose: Learn what NOT to do in feature engineering
🎯 Expected result: High CV (0.978715) but low PB (0.919028)


## Advanced Target Encoding with Over-Engineering

This implementation takes target encoding to an extreme, creating numerous variations that ultimately lead to overfitting.

In [2]:
class OverEngineeredTargetEncoder:
    """Over-engineered Target Encoder - Demonstrates how complexity can hurt
    
    This class implements numerous target encoding variations that,
    while technically sophisticated, lead to severe overfitting.
    
    ⚠️ Anti-Pattern Warning: This creates too many correlated features!
    """
    
    def __init__(self, smoothing_alpha=100, n_splits=5, random_state=42):
        self.smoothing_alpha = smoothing_alpha
        self.n_splits = n_splits
        self.random_state = random_state
        self.encoders = {}
        
    def create_multiple_smoothing_encodings(self, X, y, feature_name):
        """Create multiple target encodings with different smoothing parameters
        
        ⚠️ Problem: Creates highly correlated features that memorize training patterns
        """
        encodings = []
        
        # Multiple smoothing values - creates redundant features
        for alpha in [10, 50, 100, 200]:  # 4 different smoothing levels
            encoded = self.smooth_target_encoding_cv(X, y, feature_name, alpha)
            encoded.name = f'{feature_name}_smooth_{alpha}'
            encodings.append(encoded)
        
        return pd.concat(encodings, axis=1)
    
    def smooth_target_encoding_cv(self, X, y, feature_name, alpha):
        """CV-based smoothing target encoding"""
        cv = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        encoded_feature = np.zeros(len(X))
        
        for train_idx, valid_idx in cv.split(X, y):
            train_feature = X.iloc[train_idx][feature_name]
            train_target = y.iloc[train_idx]
            
            # Calculate smoothed encoding
            stats = pd.DataFrame({'feature': train_feature, 'target': train_target})
            category_stats = stats.groupby('feature').agg({'target': ['count', 'mean']}).reset_index()
            category_stats.columns = ['category', 'count', 'mean']
            
            global_mean = train_target.mean()
            category_stats['smoothed'] = (
                category_stats['count'] * category_stats['mean'] + alpha * global_mean
            ) / (category_stats['count'] + alpha)
            
            encoding_dict = dict(zip(category_stats['category'], category_stats['smoothed']))
            
            # Apply to validation fold
            valid_feature = X.iloc[valid_idx][feature_name]
            encoded_valid = valid_feature.map(encoding_dict).fillna(global_mean)
            encoded_feature[valid_idx] = encoded_valid
        
        return pd.Series(encoded_feature, index=X.index)
    
    def create_interaction_encodings(self, X, y, feature_pairs):
        """Create target encodings for feature interactions
        
        ⚠️ Problem: Creates complex interactions that don't generalize
        """
        interaction_features = []
        
        for feature1, feature2 in feature_pairs:
            # Create interaction feature
            interaction_name = f'{feature1}_{feature2}_interaction'
            X[interaction_name] = X[feature1].astype(str) + '_' + X[feature2].astype(str)
            
            # Encode the interaction
            encoded = self.smooth_target_encoding_cv(X, y, interaction_name, self.smoothing_alpha)
            encoded.name = f'{interaction_name}_encoded'
            interaction_features.append(encoded)
        
        return pd.concat(interaction_features, axis=1) if interaction_features else pd.DataFrame()

print("OverEngineeredTargetEncoder defined - ready for overfitting demonstration!")

OverEngineeredTargetEncoder defined - ready for overfitting demonstration!


## Excessive Feature Engineering Functions

These functions demonstrate how well-intentioned feature engineering can go wrong when taken to extremes.

In [3]:
def create_outlier_features(df):
    """Create outlier-based features
    
    ⚠️ Problem: These features memorize training set patterns
    that don't exist in the test set
    """
    print("Creating outlier features (overfitting risk!)...")
    
    outlier_features = df.copy()
    
    # Extreme alone time flag (very specific to training data)
    alone_threshold = df['Time_spent_Alone'].mean() + 2*df['Time_spent_Alone'].std()
    outlier_features['extreme_alone_flag'] = (df['Time_spent_Alone'] > alone_threshold).astype(int)
    
    # Very small social circle (again, training-specific)
    small_circle_threshold = df['Friends_circle_size'].quantile(0.05)
    outlier_features['tiny_social_circle'] = (df['Friends_circle_size'] <= small_circle_threshold).astype(int)
    
    # Extreme social media usage
    high_social_threshold = df['Post_frequency'].quantile(0.95)
    outlier_features['extreme_social_media'] = (df['Post_frequency'] >= high_social_threshold).astype(int)
    
    # Complex combination flags
    outlier_features['extreme_introvert_pattern'] = (
        (df['Time_spent_Alone'] > df['Time_spent_Alone'].quantile(0.8)) & 
        (df['Friends_circle_size'] < df['Friends_circle_size'].quantile(0.2))
    ).astype(int)
    
    outlier_features['extreme_extrovert_pattern'] = (
        (df['Post_frequency'] > df['Post_frequency'].quantile(0.8)) & 
        (df['Social_event_attendance'] > df['Social_event_attendance'].quantile(0.8))
    ).astype(int)
    
    print(f"Added 5 outlier features (high overfitting risk)")
    return outlier_features

def create_advanced_ngram_features(df):
    """Create complex N-gram features from categorical data
    
    ⚠️ Problem: Creates sparse, high-dimensional features that
    memorize training patterns rather than learning generalizable patterns
    """
    print("Creating advanced N-gram features (extreme overfitting risk!)...")
    
    ngram_features = df.copy()
    
    # Combine categorical features into text
    categorical_cols = ['Education', 'Openness', 'Marital_status']
    text_combination = df[categorical_cols].fillna('missing').apply(
        lambda row: ' '.join(row.astype(str)), axis=1
    )
    
    # Create artificial N-gram features
    # These memorize specific combinations in training data
    ngram_features['education_openness_combo'] = (
        df['Education'].astype(str) + '_' + df['Openness'].astype(str)
    )
    
    ngram_features['education_marital_combo'] = (
        df['Education'].astype(str) + '_' + df['Marital_status'].astype(str)
    )
    
    ngram_features['openness_marital_combo'] = (
        df['Openness'].astype(str) + '_' + df['Marital_status'].astype(str)
    )
    
    ngram_features['triple_combo'] = (
        df['Education'].astype(str) + '_' + 
        df['Openness'].astype(str) + '_' + 
        df['Marital_status'].astype(str)
    )
    
    # Social behavior combinations
    ngram_features['social_behavior_pattern'] = (
        (df['Post_frequency'] > df['Post_frequency'].median()).astype(str) + '_' +
        (df['Social_event_attendance'] > df['Social_event_attendance'].median()).astype(str)
    )
    
    # Time preference patterns
    ngram_features['time_preference_pattern'] = (
        (df['Time_spent_Alone'] > df['Time_spent_Alone'].median()).astype(str) + '_' +
        (df['Going_outside'] > df['Going_outside'].median()).astype(str)
    )
    
    # Complex multi-dimensional pattern
    ngram_features['complex_personality_pattern'] = (
        df['Education'].astype(str) + '_' +
        (df['Time_spent_Alone'] > df['Time_spent_Alone'].median()).astype(str) + '_' +
        (df['Friends_circle_size'] > df['Friends_circle_size'].median()).astype(str)
    )
    
    # Social anxiety combination
    ngram_features['anxiety_social_combo'] = (
        (df['Stage_fear'] > 0).astype(str) + '_' +
        (df['Drained_after_socializing'] > df['Drained_after_socializing'].median()).astype(str)
    )
    
    print(f"Added 8 N-gram combination features (extreme overfitting)")
    return ngram_features

def create_ambiversion_features(df):
    """Create features for ambiversion (balanced personality traits)
    
    ⚠️ Problem: Creates overly specific thresholds based on training data
    """
    print("Creating ambiversion features (moderate overfitting risk)...")
    
    ambi_features = df.copy()
    
    # Balanced social behavior
    social_median = df['Social_event_attendance'].median()
    alone_median = df['Time_spent_Alone'].median()
    
    ambi_features['balanced_social_alone'] = (
        (np.abs(df['Social_event_attendance'] - social_median) < social_median * 0.2) &
        (np.abs(df['Time_spent_Alone'] - alone_median) < alone_median * 0.2)
    ).astype(int)
    
    # Moderate friend circle size
    friends_q25 = df['Friends_circle_size'].quantile(0.25)
    friends_q75 = df['Friends_circle_size'].quantile(0.75)
    
    ambi_features['moderate_social_circle'] = (
        (df['Friends_circle_size'] >= friends_q25) & 
        (df['Friends_circle_size'] <= friends_q75)
    ).astype(int)
    
    # Balanced energy pattern
    energy_balance = (
        df['Going_outside'] - df['Drained_after_socializing']
    )
    ambi_features['energy_balance_score'] = energy_balance
    
    print(f"Added 3 ambiversion features")
    return ambi_features

def apply_sophisticated_imputation(df, personality_context=None):
    """Apply sophisticated missing value imputation
    
    ⚠️ Major Problem: This changes the data distribution in ways
    that don't match the test set, causing severe overfitting
    """
    print("Applying sophisticated imputation (DANGER: distribution mismatch!)...")
    
    df_imputed = df.copy()
    
    # Separate numerical and categorical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    
    # Remove non-feature columns
    numerical_cols = [col for col in numerical_cols if col not in ['id']]
    categorical_cols = [col for col in categorical_cols if col not in ['Personality']]
    
    print(f"Imputing {len(numerical_cols)} numerical and {len(categorical_cols)} categorical features")
    
    # Advanced numerical imputation with KNN
    if numerical_cols:
        print("Applying KNN imputation to numerical features...")
        
        # This is problematic: KNN assumes similar test distribution
        knn_imputer = KNNImputer(n_neighbors=5, weights='distance')
        
        # Scale features for KNN
        scaler = StandardScaler()
        df_scaled = scaler.fit_transform(df_imputed[numerical_cols])
        df_imputed_scaled = knn_imputer.fit_transform(df_scaled)
        df_imputed_rescaled = scaler.inverse_transform(df_imputed_scaled)
        
        # Replace original values
        df_imputed[numerical_cols] = df_imputed_rescaled
    
    # Personality-aware categorical imputation
    if categorical_cols and personality_context is not None:
        print("Applying personality-aware categorical imputation...")
        
        # This is highly problematic: uses target information
        for col in categorical_cols:
            if df_imputed[col].isnull().any():
                # Calculate mode by personality type (MAJOR OVERFITTING)
                personality_modes = df_imputed.groupby(personality_context)[col].apply(
                    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'
                )
                
                # Fill missing values based on personality
                for personality in personality_modes.index:
                    mask = (df_imputed[col].isnull()) & (personality_context == personality)
                    df_imputed.loc[mask, col] = personality_modes[personality]
                
                # Fill any remaining nulls with overall mode
                overall_mode = df_imputed[col].mode().iloc[0] if len(df_imputed[col].mode()) > 0 else 'Unknown'
                df_imputed[col].fillna(overall_mode, inplace=True)
    else:
        # Simple categorical imputation
        for col in categorical_cols:
            if df_imputed[col].isnull().any():
                mode_value = df_imputed[col].mode().iloc[0] if len(df_imputed[col].mode()) > 0 else 'Unknown'
                df_imputed[col].fillna(mode_value, inplace=True)
    
    print("⚠️  WARNING: Sophisticated imputation complete - high overfitting risk!")
    return df_imputed

print("Over-engineering feature functions defined!")
print("⚠️  These functions demonstrate common overfitting patterns in feature engineering")

Over-engineering feature functions defined!
⚠️  These functions demonstrate common overfitting patterns in feature engineering


## Data Loading and Baseline Preparation

In [4]:
print("=== Loading Data for Phase 4 Overfitting Demonstration ===")

# Load datasets
# train_data = pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
# test_data = pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')

train_data = pd.read_csv('/Users/osawa/kaggle/playground-series-s5e7/data/raw/train.csv')
test_data = pd.read_csv('/Users/osawa/kaggle/playground-series-s5e7/data/raw/test.csv')


print(f"Training data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

# Show missing value patterns
print("\nMissing value analysis:")
missing_train = train_data.isnull().sum()
print(missing_train[missing_train > 0])

print("\nTarget distribution:")
print(train_data['Personality'].value_counts(normalize=True))

=== Loading Data for Phase 4 Overfitting Demonstration ===
Training data: (18524, 9)
Test data: (6175, 8)

Missing value analysis:
Time_spent_Alone             1190
Stage_fear                   1893
Social_event_attendance      1180
Going_outside                1466
Drained_after_socializing    1149
Friends_circle_size          1054
Post_frequency               1264
dtype: int64

Target distribution:
Personality
Extrovert    0.739527
Introvert    0.260473
Name: proportion, dtype: float64


## Phase 4 Over-Engineered Feature Creation

This section demonstrates how excessive feature engineering leads to overfitting. We'll create 41 features from the original 7, showing how more complex doesn't mean better.

In [5]:
def create_phase4_features(train_df, test_df):
    """Create Phase 4 over-engineered features
    
    ⚠️ WARNING: This function demonstrates how NOT to do feature engineering!
    Expected result: High CV (0.978715) but catastrophic PB (0.919028)
    """
    
    print("=== Phase 4 Over-Engineering Feature Creation ===")
    print("⚠️  WARNING: This will create features that overfit severely!")
    
    # Start with original features
    feature_cols = [col for col in train_df.columns if col not in ['id', 'Personality']]
    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train = train_df['Personality'].map({'Extrovert': 1, 'Introvert': 0})
    
    print(f"Starting with {len(feature_cols)} original features")
    
    # 1. Apply sophisticated imputation (MAJOR OVERFITTING SOURCE)
    print("\n1. Applying sophisticated imputation...")
    X_train_imputed = apply_sophisticated_imputation(X_train, train_df['Personality'])
    X_test_imputed = apply_sophisticated_imputation(X_test)  # No personality context for test
    
    # 2. Create outlier features (TRAINING-SPECIFIC PATTERNS)
    print("\n2. Creating outlier features...")
    X_train_outliers = create_outlier_features(X_train_imputed)
    
    # Apply same outlier logic to test data (using training thresholds)
    X_test_outliers = X_test_imputed.copy()
    
    # Use training data statistics for test (problematic assumption)
    alone_threshold = X_train_imputed['Time_spent_Alone'].mean() + 2*X_train_imputed['Time_spent_Alone'].std()
    X_test_outliers['extreme_alone_flag'] = (X_test_imputed['Time_spent_Alone'] > alone_threshold).astype(int)
    
    small_circle_threshold = X_train_imputed['Friends_circle_size'].quantile(0.05)
    X_test_outliers['tiny_social_circle'] = (X_test_imputed['Friends_circle_size'] <= small_circle_threshold).astype(int)
    
    high_social_threshold = X_train_imputed['Post_frequency'].quantile(0.95)
    X_test_outliers['extreme_social_media'] = (X_test_imputed['Post_frequency'] >= high_social_threshold).astype(int)
    
    X_test_outliers['extreme_introvert_pattern'] = (
        (X_test_imputed['Time_spent_Alone'] > X_train_imputed['Time_spent_Alone'].quantile(0.8)) & 
        (X_test_imputed['Friends_circle_size'] < X_train_imputed['Friends_circle_size'].quantile(0.2))
    ).astype(int)
    
    X_test_outliers['extreme_extrovert_pattern'] = (
        (X_test_imputed['Post_frequency'] > X_train_imputed['Post_frequency'].quantile(0.8)) & 
        (X_test_imputed['Social_event_attendance'] > X_train_imputed['Social_event_attendance'].quantile(0.8))
    ).astype(int)
    
    # 3. Create N-gram features (SPARSE HIGH-DIMENSIONAL MEMORIZATION)
    print("\n3. Creating N-gram combination features...")
    X_train_ngrams = create_advanced_ngram_features(X_train_outliers)
    X_test_ngrams = create_advanced_ngram_features(X_test_outliers)
    
    # 4. Create ambiversion features
    print("\n4. Creating ambiversion features...")
    X_train_ambi = create_ambiversion_features(X_train_ngrams)
    X_test_ambi = create_ambiversion_features(X_test_ngrams)  # Uses test data stats - inconsistent!
    
    # 5. Apply over-engineered target encoding
    print("\n5. Applying over-engineered target encoding...")
    target_encoder = OverEngineeredTargetEncoder(smoothing_alpha=50, n_splits=5, random_state=42)
    
    # Identify categorical features (including new N-gram combinations)
    categorical_features = X_train_ambi.select_dtypes(include=['object', 'category']).columns.tolist()
    print(f"Found {len(categorical_features)} categorical features to encode")
    
    # Apply multiple smoothing encodings to each categorical feature
    for feature in categorical_features:
        print(f"  Over-encoding {feature}...")
        
        # Create multiple smoothing variations (REDUNDANT FEATURES)
        multi_encodings = target_encoder.create_multiple_smoothing_encodings(
            X_train_ambi, y_train, feature
        )
        X_train_ambi = pd.concat([X_train_ambi, multi_encodings], axis=1)
        
        # Apply to test data (simplified)
        for alpha in [10, 50, 100, 200]:
            # Calculate encoding from training data
            train_stats = pd.DataFrame({
                'feature': X_train_ambi[feature],
                'target': y_train
            })
            category_stats = train_stats.groupby('feature').agg({'target': ['count', 'mean']}).reset_index()
            category_stats.columns = ['category', 'count', 'mean']
            
            global_mean = y_train.mean()
            category_stats['smoothed'] = (
                category_stats['count'] * category_stats['mean'] + alpha * global_mean
            ) / (category_stats['count'] + alpha)
            
            encoding_dict = dict(zip(category_stats['category'], category_stats['smoothed']))
            
            # Apply to test
            test_encoded = X_test_ambi[feature].map(encoding_dict).fillna(global_mean)
            X_test_ambi[f'{feature}_smooth_{alpha}'] = test_encoded
    
    # 6. Final assembly
    print("\n6. Final feature assembly...")
    
    # Create final datasets
    train_final = pd.concat([
        train_df[['id', 'Personality']], 
        X_train_ambi
    ], axis=1)
    
    test_final = pd.concat([
        test_df[['id']],
        X_test_ambi
    ], axis=1)
    
    print(f"\n⚠️  Phase 4 over-engineering complete!")
    print(f"Training data: {train_final.shape}")
    print(f"Test data: {test_final.shape}")
    print(f"Original features: {len(feature_cols)}")
    print(f"Final features: {train_final.shape[1] - 2}")  # Exclude id, Personality
    print(f"Feature explosion: {((train_final.shape[1] - 2) / len(feature_cols) - 1) * 100:.1f}% increase!")
    print(f"\n🚨 CRITICAL WARNING: This will severely overfit!")
    
    return train_final, test_final

# Create the over-engineered features
train_overengineered, test_overengineered = create_phase4_features(train_data, test_data)

=== Phase 4 Over-Engineering Feature Creation ===
⚠️  WARNING: This will create features that overfit severely!
Starting with 7 original features

1. Applying sophisticated imputation...
Applying sophisticated imputation (DANGER: distribution mismatch!)...
Imputing 5 numerical and 2 categorical features
Applying KNN imputation to numerical features...
Applying personality-aware categorical imputation...
⚠️  WARNING: Sophisticated imputation complete - high overfitting risk!
Applying sophisticated imputation (DANGER: distribution mismatch!)...
Imputing 5 numerical and 2 categorical features
Applying KNN imputation to numerical features...
⚠️  WARNING: Sophisticated imputation complete - high overfitting risk!

2. Creating outlier features...
Creating outlier features (overfitting risk!)...
Added 5 outlier features (high overfitting risk)

3. Creating N-gram combination features...
Creating advanced N-gram features (extreme overfitting risk!)...


KeyError: "None of [Index(['Education', 'Openness', 'Marital_status'], dtype='object')] are in the [columns]"

## Model Training and Cross-Validation

Now we'll train our ensemble on the over-engineered features and observe the dangerously high CV score that doesn't translate to test performance.

In [6]:
def create_overfitting_ensemble():
    """Create ensemble model for overfitting demonstration"""
    
    models = [
        ('lgb', lgb.LGBMClassifier(
            objective='binary',
            num_leaves=31,
            learning_rate=0.02,
            n_estimators=1500,
            random_state=42,
            verbosity=-1
        )),
        ('xgb', xgb.XGBClassifier(
            objective='binary:logistic',
            max_depth=6,
            learning_rate=0.02,
            n_estimators=1500,
            random_state=42,
            verbosity=0
        )),
        ('cat', CatBoostClassifier(
            objective='Logloss',
            depth=6,
            learning_rate=0.02,
            iterations=1500,
            random_seed=42,
            verbose=False
        )),
        ('lr', LogisticRegression(
            random_state=42,
            max_iter=1000
        ))
    ]
    
    return VotingClassifier(estimators=models, voting='soft')

print("=== Phase 4 Model Training (Overfitting Demonstration) ===")

# Prepare training data
feature_cols = [col for col in train_overengineered.columns 
               if col not in ['id', 'Personality']]

X_train = train_overengineered[feature_cols]
y_train = train_overengineered['Personality'].map({'Extrovert': 1, 'Introvert': 0})

print(f"Training with {len(feature_cols)} over-engineered features")
print(f"Training samples: {len(X_train)}")
print(f"Feature-to-sample ratio: 1:{len(X_train)/len(feature_cols):.1f}")

# Handle missing values and encode categorical features
X_train_processed = X_train.copy()

# Label encode remaining categorical features
label_encoders = {}
for col in feature_cols:
    if X_train_processed[col].dtype == 'object':
        le = LabelEncoder()
        # Fit on all unique values
        all_values = pd.concat([X_train_processed[col], test_overengineered[col]]).astype(str)
        le.fit(all_values)
        X_train_processed[col] = le.transform(X_train_processed[col].astype(str))
        label_encoders[col] = le

# Fill remaining missing values
X_train_processed = X_train_processed.fillna(0)

print(f"\nEncoded {len(label_encoders)} categorical features")

# Create and evaluate ensemble
print("\nCreating overfitting ensemble...")
ensemble_model = create_overfitting_ensemble()

print("\nPerforming cross-validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(ensemble_model, X_train_processed, y_train, cv=cv, scoring='accuracy')

print(f"\n📊 Cross-Validation Results (MISLEADING!):")
print(f"Individual fold scores: {cv_scores}")
print(f"Mean CV Score: {cv_scores.mean():.6f}")
print(f"Standard Deviation: {cv_scores.std():.6f}")

print(f"\n🚨 CRITICAL ANALYSIS:")
print(f"CV Score: {cv_scores.mean():.6f} (DANGEROUSLY HIGH!)")
print(f"Expected PB Score: ~0.919028 (CATASTROPHIC FAILURE!)")
print(f"Expected CV-PB Gap: ~-0.060 (85x worse than healthy models)")
print(f"\n⚠️  The high CV score is MISLEADING - this model will fail on test data!")

=== Phase 4 Model Training (Overfitting Demonstration) ===


NameError: name 'train_overengineered' is not defined

## Final Predictions and Overfitting Analysis

In [7]:
print("=== Final Model Training and Prediction ===")

# Train final model
print("Training final overfitting model...")
ensemble_model.fit(X_train_processed, y_train)

# Prepare test data
X_test = test_overengineered[feature_cols]
X_test_processed = X_test.copy()

# Apply same preprocessing to test data
for col in feature_cols:
    if col in label_encoders:
        X_test_processed[col] = label_encoders[col].transform(X_test_processed[col].astype(str))

X_test_processed = X_test_processed.fillna(0)
test_ids = test_overengineered['id']

print(f"Test samples: {len(X_test_processed)}")

# Generate predictions
print("\nGenerating predictions...")
test_probabilities = ensemble_model.predict_proba(X_test_processed)[:, 1]
test_predictions = (test_probabilities > 0.5).astype(int)

# Create submission
submission_df = pd.DataFrame({
    'id': test_ids,
    'Personality': ['Extrovert' if pred == 1 else 'Introvert' for pred in test_predictions]
})

# Analyze predictions
extrovert_count = np.sum(test_predictions == 1)
introvert_count = np.sum(test_predictions == 0)
avg_confidence = np.mean(np.maximum(test_probabilities, 1 - test_probabilities))

print(f"\n📊 Prediction Statistics:")
print(f"Extrovert: {extrovert_count} ({extrovert_count/len(test_predictions)*100:.1f}%)")
print(f"Introvert: {introvert_count} ({introvert_count/len(test_predictions)*100:.1f}%)")
print(f"Average Confidence: {avg_confidence:.4f}")

print(f"\nSubmission preview:")
print(submission_df.head(10))

# Save submission (uncomment if needed)
# submission_df.to_csv('phase4_overfitting_submission.csv', index=False)
# print(f"\nSubmission saved as 'phase4_overfitting_submission.csv'")

=== Final Model Training and Prediction ===
Training final overfitting model...


NameError: name 'ensemble_model' is not defined

## Feature Importance Analysis - Understanding the Overfitting

In [8]:
print("=== Feature Importance Analysis (Overfitting Patterns) ===")

# Extract feature importance from LightGBM
lgb_model = ensemble_model.named_estimators_['lgb']
feature_importance = lgb_model.feature_importances_

# Create importance DataFrame
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features (Many are overfitting artifacts):")
top_features = importance_df.head(20)
for idx, row in top_features.iterrows():
    feature_type = "📊 Original" if row['feature'] in ['Time_spent_Alone', 'Friends_circle_size', 'Post_frequency', 
                                                        'Social_event_attendance', 'Going_outside', 
                                                        'Drained_after_socializing', 'Stage_fear', 
                                                        'Education', 'Openness', 'Marital_status'] else "🚨 Engineered"
    print(f"{feature_type} {row['feature']}: {row['importance']:.4f}")

# Analyze feature categories
original_features = importance_df[importance_df['feature'].isin([
    'Time_spent_Alone', 'Friends_circle_size', 'Post_frequency', 
    'Social_event_attendance', 'Going_outside', 'Drained_after_socializing', 
    'Stage_fear', 'Education', 'Openness', 'Marital_status'
])]

outlier_features = importance_df[importance_df['feature'].str.contains('extreme|tiny|balanced|moderate')]
ngram_features = importance_df[importance_df['feature'].str.contains('combo|pattern')]
encoded_features = importance_df[importance_df['feature'].str.contains('smooth')]

print(f"\n🔍 Feature Category Analysis:")
print(f"Original features (10): Avg importance {original_features['importance'].mean():.4f}")
print(f"Outlier features ({len(outlier_features)}): Avg importance {outlier_features['importance'].mean():.4f}")
print(f"N-gram features ({len(ngram_features)}): Avg importance {ngram_features['importance'].mean():.4f}")
print(f"Target encoded features ({len(encoded_features)}): Avg importance {encoded_features['importance'].mean():.4f}")

print(f"\n🚨 Overfitting Indicators:")
print(f"1. Feature explosion: {len(feature_cols)} features (vs 7 original)")
print(f"2. High importance on engineered features: {(len(importance_df) - len(original_features))} artificial")
print(f"3. Complex interaction features getting high importance")
print(f"4. Multiple redundant encodings of same information")

# Show most problematic features
print(f"\n🎯 Most Problematic Features (Overfitting Sources):")
problematic = importance_df[
    importance_df['feature'].str.contains('extreme|combo|smooth|pattern')
].head(10)

for idx, row in problematic.iterrows():
    print(f"⚠️  {row['feature']}: {row['importance']:.4f}")

=== Feature Importance Analysis (Overfitting Patterns) ===


NameError: name 'ensemble_model' is not defined

## Critical Analysis: Why Phase 4 Failed

### The Overfitting Disaster

Phase 4 represents a perfect case study of how sophisticated feature engineering can backfire spectacularly:

**Performance Metrics:**
- **CV Score**: 0.978715 ± 0.000933 (highest ever achieved)
- **PB Score**: 0.919028 (catastrophic failure)
- **CV-PB Gap**: -0.059687 (85x worse than successful models)

### Root Causes of Failure

#### 1. Feature Explosion (17 → 41 features)
- **Problem**: 141% increase in feature count
- **Impact**: Curse of dimensionality with only 24,430 training samples
- **Ratio**: ~595 samples per feature (dangerously low)

#### 2. Sophisticated Imputation Backfire
- **KNN Imputation**: Assumes test distribution matches training
- **Personality-aware Imputation**: Uses target information (data leakage)
- **Distribution Change**: Artificially modifies natural data patterns

#### 3. Target Encoded N-grams
- **Sparse Features**: Creates many rare category combinations
- **Memorization**: Learns training-specific patterns
- **High Dimensionality**: 8 complex categorical features with 4 encodings each

#### 4. Training-Specific Outlier Detection
- **Static Thresholds**: Based on training data statistics
- **Distribution Mismatch**: Test data has different outlier patterns
- **Overfitting**: Learns training set noise as signal

### The CV-PB Gap Phenomenon

```python
# Comparison of CV-PB Gaps
Phase 2b (Success):  CV 0.968905 → PB 0.975708 (Gap: +0.006803)
Phase 3 (Success):   CV 0.976404 → PB 0.975708 (Gap: -0.000696)
Phase 4 (Failure):   CV 0.978715 → PB 0.919028 (Gap: -0.059687)
```

The massive negative gap in Phase 4 is a clear indicator of severe overfitting.

### Key Lessons Learned

1. **Simplicity Wins**: 17 features (Phase 3) outperformed 41 features (Phase 4)
2. **CV Can Mislead**: High CV scores don't guarantee test performance
3. **Data Distribution Matters**: Imputation must preserve natural patterns
4. **Feature Quality > Quantity**: Better to have fewer, robust features
5. **Beware of Complexity**: Sophisticated methods can memorize noise

### Best Practices to Avoid Overfitting

1. **Monitor CV-PB Gap**: Healthy gap should be ±0.01
2. **Limit Feature Count**: Keep features < 20 for this dataset size
3. **Simple Imputation**: Use zero-filling or simple statistics
4. **Validate Incrementally**: Add features one at a time
5. **Preserve Distribution**: Don't artificially change data patterns

### Conclusion

Phase 4 demonstrates that in machine learning competitions:
- **More complex ≠ Better performance**
- **CV scores can be dangerously misleading**
- **Overfitting can destroy months of work**
- **Simple approaches often win**

This failure, while painful, provides invaluable lessons for future feature engineering efforts.

In [ ]:
print("🚨 Phase 4: Overfitting Demonstration Complete!")
print("="*60)
print("❌ CV Score: 0.978715 (misleadingly high)")
print("❌ PB Score: 0.919028 (catastrophic failure)")
print("❌ CV-PB Gap: -0.059687 (85x worse than healthy models)")
print("❌ Feature Explosion: 41 features (586% of original)")
print("="*60)
print("📚 CRITICAL LESSONS:")
print("1. High CV scores can be completely misleading")
print("2. Feature engineering complexity has diminishing returns")
print("3. Data distribution preservation is crucial")
print("4. Simple approaches often outperform complex ones")
print("5. CV-PB gap monitoring is essential for overfitting detection")
print("="*60)
print("⚠️  Use this as a reference for what NOT to do in feature engineering!")